In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from pathlib import Path

# 1. Flexible Paths Configuration
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

HISTORICAL_ELO_PATH = BASE_DIR / "data" / "processed" / "historical_with_elo.csv"
FIXTURES_PATH = BASE_DIR / "data" / "processed" / "model_input_fixtures.csv"
OUTPUT_PREDICTIONS_PATH = BASE_DIR / "data" / "processed" / "knockout_predictions.csv"

print("[INFO] Loading historical data for AI training...")
df_history = pd.read_csv(HISTORICAL_ELO_PATH)

# 2. Preparing Training Data (The Classroom for our AI)
# We calculate the historical Elo difference
df_history['elo_diff'] = df_history['home_elo_before'] - df_history['away_elo_before']

# Target variable: 1 if Home Team wins, 0 if Away Team wins or draws (strict knockout logic)
df_history['target'] = np.where(df_history['home_score'] > df_history['away_score'], 1, 0)

# We drop initial NaN values (first matches of a team where Elo was still forming)
df_history = df_history.dropna(subset=['elo_diff'])

X_train = df_history[['elo_diff']]
y_train = df_history['target']

# 3. Training the Random Forest Algorithm
print("[INFO] Training Random Forest model on 150 years of match history...")
# We restrict depth to 5 to prevent overfitting
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print("[SUCCESS] Artificial Intelligence is fully trained!")

# 4. Predicting the World Cup Fixtures
print("[INFO] Generating predictions for upcoming World Cup matches...")
df_fixtures = pd.read_csv(FIXTURES_PATH)
X_test = df_fixtures[['elo_diff']]

# Get raw probabilities from the historical Elo model
raw_probs = model.predict_proba(X_test)
df_fixtures['home_win_prob'] = raw_probs[:, 1]
df_fixtures['away_win_prob'] = raw_probs[:, 0]

# 5. The Magic Touch: Form Index Blending (FBref Stats Integration)
# We slightly boost or penalize probabilities based on real-time tournament form
# e.g., higher possession diff and goal diff slightly increase odds
df_fixtures['form_modifier'] = (df_fixtures['poss_diff'] * 0.002) + (df_fixtures['gls_diff'] * 0.02)

# Apply modifier and normalize so probabilities still add up to 100%
df_fixtures['home_win_prob_adjusted'] = df_fixtures['home_win_prob'] + df_fixtures['form_modifier']
df_fixtures['away_win_prob_adjusted'] = df_fixtures['away_win_prob'] - df_fixtures['form_modifier']

# Clip values to ensure they stay between 1% and 99%
df_fixtures['home_win_prob_adjusted'] = df_fixtures['home_win_prob_adjusted'].clip(0.01, 0.99)
df_fixtures['away_win_prob_adjusted'] = 1.0 - df_fixtures['home_win_prob_adjusted']

# Convert to readable percentages
df_fixtures['Home_Advancement_Chance'] = (df_fixtures['home_win_prob_adjusted'] * 100).round(1).astype(str) + '%'
df_fixtures['Away_Advancement_Chance'] = (df_fixtures['away_win_prob_adjusted'] * 100).round(1).astype(str) + '%'

# Save Final Predictions
final_output = df_fixtures[['date', 'home_team', 'away_team', 'Home_Advancement_Chance', 'Away_Advancement_Chance']]
final_output.to_csv(OUTPUT_PREDICTIONS_PATH, index=False)

print(" PREDICTIONS GENERATED SUCCESSFULLY")
print("="*60)
print("Here are the mathematical odds for the Knockout Stage matchups:")
print("-" * 60)
print(final_output.to_string(index=False))

[INFO] Loading historical data for AI training...
[INFO] Training Random Forest model on 150 years of match history...
[SUCCESS] Artificial Intelligence is fully trained!
[INFO] Generating predictions for upcoming World Cup matches...

 PREDICTIONS GENERATED SUCCESSFULLY
Here are the mathematical odds for the Knockout Stage matchups:
------------------------------------------------------------
      date     home_team              away_team Home_Advancement_Chance Away_Advancement_Chance
2026-07-01       England               DR Congo                   85.3%                   14.7%
2026-07-01       Belgium                Senegal                   51.3%                   48.7%
2026-07-01 United States Bosnia and Herzegovina                   77.9%                   22.1%
2026-07-02         Spain                Austria                   76.6%                   23.4%
2026-07-02      Portugal                Croatia                   58.1%                   41.9%
2026-07-02   Switzerland   